# Appendix B — `serm` Package Reference

This appendix is **auto-generated**: it imports the `serm` package and renders the
signature and docstring of every public function in each module, followed by one
tiny runnable example per module.  Re-execute this notebook to regenerate the
reference after editing the package.

The `serm` package is the numpy / scipy / matplotlib re-implementation of the
algorithms in Michael Honeychurch's *Simulating Electrochemical Reactions in
Mathematica* (SERM).

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

# %matplotlib inline embeds figures and makes plt.show() a harmless no-op under headless (Agg) execution
%matplotlib inline

import importlib, inspect, textwrap
import numpy as np
import matplotlib.pyplot as plt

import serm
print("serm package loaded from:", os.path.dirname(serm.__file__))

serm package loaded from: /home/nsg/Dropbox/Files/Git/serm-python/serm


## B.1 Auto-rendered API reference

The helper below walks a module, finds every public (non-underscore) function
that is *defined in that module*, and prints its signature and docstring.

In [2]:
from IPython.display import Markdown, display

def document_module(module_name):
    """Render signature + docstring for each public function defined in a module."""
    mod = importlib.import_module(module_name)
    # explicit name lists (functions defined in the module itself)
    members = []
    for name, obj in vars(mod).items():
        if name.startswith("_"):
            continue
        if inspect.isfunction(obj) and getattr(obj, "__module__", "") == module_name:
            members.append((name, obj))
    members.sort(key=lambda kv: kv[0])

    lines = [f"## `{module_name}`", ""]
    doc = inspect.getdoc(mod)
    if doc:
        lines += [doc.split("\n\n")[0], ""]
    for name, obj in members:
        sig = inspect.signature(obj)
        lines.append(f"### `{name}{sig}`")
        lines.append("")
        fdoc = inspect.getdoc(obj) or "*(no docstring)*"
        lines.append("```")
        lines.append(fdoc)
        lines.append("```")
        lines.append("")
    display(Markdown("\n".join(lines)))

MODULES = [
    "serm",
    "serm.tridiagonal",
    "serm.filters",
    "serm.grids",
    "serm.plotting",
    "serm.waveforms",
    "serm.echem",
    "serm.kinetics",
]
for m in MODULES:
    document_module(m)

## `serm`

serm -- Simulating Electrochemical Reactions in Python.

### `cottrell_dimensionless(n)`

```
Analytical Cottrell dimensionless current ``1/sqrt(pi*tau)``.

This is the *dimensionless* Cottrell transient on the chapter's ``tau``
grid (``tau = (k-1)/(n-1)``), taking the number of time steps ``n`` as its
only argument.  It is distinct from :func:`serm.echem.cottrell_current`,
which evaluates the dimensional Cottrell current ``i(t)`` in amperes.

Port of ``z = Table[-1/Sqrt[Pi (k-1)/(n-1)], {k, 2, n-1}]`` from
``ExplicitFD.nb``.  For a potential step to the diffusion-limited region,
the flux at a planar electrode follows the Cottrell form; in the chapter's
dimensionless variables (``tau = (k-1)/(n-1)``) this is ``1/sqrt(pi*tau)``.

Returns the magnitude for each time index ``k = 0 .. n-1``; ``tau = 0``
gives ``inf`` and is returned as nan.

Parameters
----------
n : int

Returns
-------
ndarray, shape (n,)
```

### `electrode_current(c, D_M)`

```
Dimensionless current transient at the electrode.

Port of the ``i1`` expression in ``ExplicitFD.nb``::

    i1 = (-4 c[2] + c[3]) * 0.5 * Sqrt[D_M (n-1)]   (1-based indices)

Per time slice (column ``k``), the flux at the electrode is approximated by
a one-sided finite difference of the concentration gradient.  With the
surface concentration pinned to zero (``c[0] = 0``), the 3-point one-sided
derivative ``(-3 c[0] + 4 c[1] - c[2]) / (2 dx)`` reduces to
``(4 c[1] - c[2]) / (2 dx)``.  Multiplying the gradient by the dimensionless
factor ``Sqrt(D_M (n-1)) = 1/dx`` converts grid units to the dimensionless
current (the chapter's ``(tn/D)^{1/2}`` scaling).

Returns the magnitude (positive) of the current for each time column.

Parameters
----------
c : ndarray, shape (m, n)
D_M : float

Returns
-------
ndarray, shape (n,)
    Dimensionless current at each time index; ``i[0]`` (t=0) is set to nan.
```

### `explicit_solve(c, D_M)`

```
Advance the concentration grid in time with the explicit FD scheme.

Port of ``explicitSolve`` / ``explicitSolve2`` from ``ExplicitFD.nb``.

The forward-difference (fully explicit) discretisation of Fick's second law

    dc/dt = d^2 c / dx^2     (dimensionless)

is

    c[j, k] = D_M * c[j-1, k-1] + (1 - 2*D_M) * c[j, k-1]
              + D_M * c[j+1, k-1]

where ``D_M = dt / dx^2`` is the dimensionless model diffusion coefficient.
This is the exact update in the original procedural ``explicitSolve``; here
we vectorise the spatial sweep (the interior of each column) as in the
original ``explicitSolve2`` (``ListCorrelate[{D, 1-2D, D}, ...]``).

The scheme is stable only for ``D_M <= 0.5``.

Parameters
----------
c : ndarray, shape (m, n)
    Grid from :func:`serm.grids.make_grid`, with IC/BCs already applied.
    Modified in place and also returned.
D_M : float
    Dimensionless model diffusion coefficient.

Returns
-------
ndarray, shape (m, n)
    The filled grid.
```


## `serm.tridiagonal`

Tridiagonal linear-system solvers (Thomas algorithm).

### `tridiag_solve(x, y, z, b)`

```
Solve a tridiagonal system ``A . u == b`` with the Thomas algorithm.

Direct port of the original ``TridiagSolver`` (no pivoting).  This is a
forward sweep (eliminate the sub-diagonal) followed by a back substitution.

Parameters
----------
x : array_like, shape (n-1,)
    Sub-diagonal of ``A`` (entries below the main diagonal).
y : array_like, shape (n,)
    Main diagonal of ``A``.
z : array_like, shape (n-1,)
    Super-diagonal of ``A`` (entries above the main diagonal).
b : array_like, shape (n,)
    Right-hand side.

Returns
-------
numpy.ndarray, shape (n,)
    Solution vector ``u``.

Notes
-----
Like the original, this performs *no pivoting*: a zero pivot produces
``inf``/``nan`` (the original warns "Infinity introduced if pivot becomes
zero").  For diagonally dominant systems -- the usual case for implicit
finite-difference diffusion problems -- it is stable.
```

### `tridiag_solve_banded(x, y, z, b)`

```
Solve the same system via :func:`scipy.linalg.solve_banded`.

This is the recommended production path: SciPy calls LAPACK's banded solver
(``gtsv``-class routines via the banded interface), which *does* pivot and
is more robust than the bare Thomas algorithm.  Same argument convention as
:func:`tridiag_solve`.

Returns
-------
numpy.ndarray, shape (n,)
```


## `serm.filters`

Smoothing filters for noisy simulated data.

### `convolution_filter(data, length)`

```
Gaussian smoothing filter.

Port of ``ConvolutionFilter[data, len]``.  The original applies the kernel
with ``ListConvolve[kern, data, {-1-len, 1+len}]``; the overlap spec
``{-1-len, 1+len}`` makes the convolution *cyclic* (wrap-around) and returns
an output the same length as the input, centred on the kernel.  We match
that here with ``mode='wrap'`` semantics via :func:`numpy.convolve` on a
periodically extended signal.

Parameters
----------
data : array_like, 1-D
length : int
    Half-width of the kernel (kernel has ``2*length + 1`` points).

Returns
-------
numpy.ndarray, same length as ``data``.
```

### `gaussian_kernel(length)`

```
Return the normalised Gaussian kernel used by ``ConvolutionFilter``.

``Table[Exp[-k^2/100], {k, -length, length}]`` normalised to unit sum.
Has ``2*length + 1`` points.
```

### `moving_average(data, n)`

```
``n``-point moving average.

Port of ``MovingAve[list, n]``.  The original uses
``ListCorrelate[Table[1., {n}], list] / n``, which is a *valid*-mode
correlation with a flat kernel: the output has ``len(data) - n + 1``
points, each the mean of ``n`` consecutive input points.

Parameters
----------
data : array_like, 1-D
n : int
    Window width (positive).

Returns
-------
numpy.ndarray, shape (len(data) - n + 1,)
```


## `serm.grids`

Finite-difference grid helpers for the SERM diffusion simulations.

### `dx_dimensionless(D_M, n)`

```
Dimensionless space step ``dx = 1 / sqrt(D_M (n-1))``.

Derived in the chapter: with dimensionless time step ``dt = 1/(n-1)`` and
``D_M = dt / dx^2``, we get ``dx = sqrt(dt / D_M) = 1/sqrt(D_M (n-1))``.
```

### `make_grid(m, n)`

```
Build the ``m x n`` concentration grid with IC/BCs applied.

Port of ``makeGrid[m, n]`` from ``ExplicitFD.nb``.  Returns a float array
``c`` with ``c[:, 0] = 1`` (initial condition), ``c[0, 1:] = 0`` (electrode
boundary for t > 0) and ``c[-1, 1:] = 1`` (bulk boundary).

Parameters
----------
m : int
    Number of space points (row 0 = electrode, row m-1 = bulk).
n : int
    Number of time points (column 0 = initial condition).

Returns
-------
numpy.ndarray, shape (m, n)
```

### `space_points(D_M, n)`

```
Number of spatial grid points ``m`` for ``n`` time points.

Port of ``m = 1 + Ceiling[6 Sqrt[D_M (n-1)]]`` from ``ExplicitFD.nb``.

The dimensionless diffusion-layer extent is fixed at ``x_dv = 6`` (six
diffusion lengths -- effectively semi-infinite over the experiment), and the
number of space steps needed to reach it is ``6 Sqrt(D_M (n-1))`` because
the dimensionless space step is ``dx = 1 / Sqrt(D_M (n-1))`` (see
:func:`dx_dimensionless`).

Parameters
----------
D_M : float
    Model (dimensionless) diffusion coefficient ``D_M = D*dt/dx^2``.
n : int
    Number of time grid points.

Returns
-------
int
```


## `serm.plotting`

Shared matplotlib helpers for the SERM Python notebooks.

### `animate_profiles(c, x, step=4, interval=60)`

```
Return a FuncAnimation of the evolving concentration profile.

Parameters
----------
c : ndarray, shape (m, n)
x : ndarray, shape (m,)
step : int
    Plot every ``step``-th time slice.
interval : int
    Delay between frames (ms).
```

### `plot_current(tau, i_sim, i_cottrell=None, ax=None)`

```
Plot the dimensionless current transient, optionally with Cottrell.
```

### `plot_profiles(c, taus, x, ax=None)`

```
Plot concentration vs. distance at several dimensionless times.

Parameters
----------
c : ndarray, shape (m, n)
    Concentration grid (space x time).
taus : sequence of float
    Dimensionless times (0..1) at which to draw a profile.
x : ndarray, shape (m,)
    Dimensionless distance coordinate.
ax : matplotlib axis, optional
```

### `plot_surface(c, x, ax=None)`

```
3-D surface of concentration over distance and time (replaces ListPlot3D).

Parameters
----------
c : ndarray, shape (m, n)
x : ndarray, shape (m,)
    Dimensionless distance.
```


## `serm.waveforms`

Applied-potential waveform generators for the voltammetry chapters.

### `ac_superposition(t, E_dc, amplitude, frequency, phase=0.0)`

```
Add a sinusoidal AC perturbation to a DC waveform (AC voltammetry).

``E(t) = E_dc(t) + amplitude * sin(2 pi frequency t + phase)``.

Parameters
----------
t : array_like
    Time samples (s).
E_dc : array_like
    DC waveform sampled at ``t`` (V) -- e.g. the output of
    :func:`linear_sweep` or :func:`cyclic_sweep`.
amplitude : float
    AC amplitude (V).
frequency : float
    AC frequency (Hz).
phase : float, optional
    Phase offset (rad).

Returns
-------
numpy.ndarray
    The superposed potential, same shape as ``t``.
```

### `cyclic_sweep(E_start, E_switch, v, n_points, E_end=None)`

```
Triangular cyclic-voltammetry waveform.

Sweeps linearly from ``E_start`` to the switching potential ``E_switch`` and
back to ``E_end`` (default: ``E_start``) at constant rate ``v``.  This is the
standard CV excitation of SERM Chapter 5.

Parameters
----------
E_start : float
    Initial potential (V).
E_switch : float
    Switching (vertex) potential (V).
v : float
    Sweep rate magnitude (V/s).
n_points : int
    Total number of samples over the whole forward+reverse sweep (>= 3).
E_end : float, optional
    Potential at the end of the reverse sweep; defaults to ``E_start``.

Returns
-------
t, E : numpy.ndarray, shape (n_points,)
    Time (s) and potential (V) arrays. ``t`` is uniformly spaced; ``E`` is
    the triangular ramp.
```

### `dimensionless_sweep_rate(v, n_electrons=1, temperature=298.15)`

```
Dimensionless sweep parameter ``sigma = n F v / (R T)`` (units 1/s).

From SERM Ch. 5: the CV problem is non-dimensionalised in time by ``sigma``,
so the dimensionless potential axis is ``sigma * t`` (see
``Chapters/chapter5.nb``).

Parameters
----------
v : float
    Sweep rate (V/s).
n_electrons : int
    Number of electrons transferred.
temperature : float
    Temperature (K); default 298.15.

Returns
-------
float
```

### `linear_sweep(E_start, E_end, v, n_points)`

```
Single linear potential ramp ``E(t) = E_start +/- v t``.

Parameters
----------
E_start, E_end : float
    Initial and final potential (V).
v : float
    Sweep rate magnitude (V/s); the sign of the ramp is taken from
    ``sign(E_end - E_start)``.
n_points : int
    Number of samples (>= 2).

Returns
-------
t, E : numpy.ndarray, shape (n_points,)
    Time (s) and potential (V) arrays.
```

### `nernst_theta(E, E0=0.0, n_electrons=1, temperature=298.15)`

```
Nernstian surface ratio ``theta = exp[n F (E - E0)/(R T)]``.

For a reversible couple ``O + n e- <=> R`` the surface concentration ratio
is ``c_O / c_R = exp[n F (E - E0)/(R T)]``; the diffusion-limited reversible
voltammogram uses ``c_O(surface) = 1/(1 + 1/theta)`` (SERM Ch. 5 boundary
condition ``xi[y]/(1 + xi[y])`` with ``xi = 1/theta``).

Parameters
----------
E : array_like
    Applied potential (V).
E0 : float
    Formal potential (V).
n_electrons : int
    Electrons transferred.
temperature : float
    Temperature (K).

Returns
-------
numpy.ndarray
    ``theta`` at each potential.
```

### `potential_step(E_initial, E_final, t_total, n_points, t_step=0.0)`

```
Single potential step (chronoamperometry / Cottrell experiment).

``E = E_initial`` for ``t < t_step`` and ``E = E_final`` for ``t >= t_step``.

Parameters
----------
E_initial, E_final : float
    Potentials before and after the step (V).
t_total : float
    Total experiment duration (s).
n_points : int
    Number of samples (>= 2).
t_step : float, optional
    Time of the step (s); default 0 (step at the start).

Returns
-------
t, E : numpy.ndarray, shape (n_points,)
```

### `pulse_train(base_levels, t_total, n_points)`

```
Staircase / pulse-train waveform from a sequence of potential levels.

The experiment time is split into ``len(base_levels)`` equal segments, each
held at the corresponding level.  Use this for staircase CV and the
base waveforms of normal-/differential-pulse voltammetry (Chapter 8).

Parameters
----------
base_levels : sequence of float
    Potential held during each successive equal-duration segment (V).
t_total : float
    Total duration (s).
n_points : int
    Number of samples (>= len(base_levels)).

Returns
-------
t, E : numpy.ndarray, shape (n_points,)
```


## `serm.echem`

Independent analytic reference results for validation.

### `cottrell_current(t, n, A, D, c_bulk)`

```
Cottrell equation -- current after a diffusion-limited potential step.

.. math::
    i(t) = \frac{n F A \sqrt{D}\, c^*}{\sqrt{\pi t}}

(Bard & Faulkner, *Electrochemical Methods*, 2nd ed., potential-step /
chronoamperometry chapter.)  Validates Chapters 2 and 8.

Parameters
----------
t : array_like
    Time (s), ``> 0``.
n : int
    Electrons transferred.
A : float
    Electrode area (cm^2 if ``D`` is cm^2/s and ``c_bulk`` mol/cm^3).
D : float
    Diffusion coefficient (cm^2/s).
c_bulk : float
    Bulk concentration of the electroactive species (mol/cm^3).

Returns
-------
numpy.ndarray
    Current (A). ``t = 0`` yields ``inf``.

Notes
-----
Keep units self-consistent: with ``A`` in cm^2, ``D`` in cm^2/s and
``c_bulk`` in mol/cm^3 the result is in amperes.
```

### `koutecky_levich_current(i_kinetic, n, A, D, c_bulk, omega, nu)`

```
Koutecky--Levich combination of kinetic and mass-transport currents.

.. math::
    \frac{1}{i} = \frac{1}{i_k} + \frac{1}{i_L}

where ``i_L`` is the Levich current (Bard & Faulkner, *Electrochemical
Methods*, 2nd ed., rotating-disk / hydrodynamic-methods chapter).  Returns
the measured current ``i`` for a given kinetic current ``i_k``.

Parameters
----------
i_kinetic : float
    Kinetically limited current ``i_k`` (A).
n, A, D, c_bulk, omega, nu :
    As in :func:`levich_current`.

Returns
-------
float
    Net current ``i`` (A).
```

### `levich_current(n, A, D, c_bulk, omega, nu)`

```
Levich equation -- limiting current at a rotating disk electrode.

.. math::
    i_L = 0.620\, n F A D^{2/3} \nu^{-1/6} \omega^{1/2} c^*

(Bard & Faulkner, *Electrochemical Methods*, 2nd ed., rotating-disk /
hydrodynamic-methods chapter; ``omega`` in rad/s.)  Validates Chapter 14.

Parameters
----------
n : int
    Electrons transferred.
A : float
    Electrode area (cm^2).
D : float
    Diffusion coefficient (cm^2/s).
c_bulk : float
    Bulk concentration (mol/cm^3).
omega : float
    Angular rotation rate (rad/s).
nu : float
    Kinematic viscosity (cm^2/s).

Returns
-------
float
    Levich limiting current (A).
```

### `randles_sevcik_peak_current(n, A, D, c_bulk, v, temperature=298.15, reversible=True, alpha=0.5)`

```
Peak current of a (reversible) linear-sweep / cyclic voltammogram.

Reversible (Randles--Sevcik):

.. math::
    i_p = 0.4463\, n F A c^* \sqrt{\frac{n F v D}{R T}}

At 298.15 K this is the familiar ``i_p = 2.69e5 n^{3/2} A D^{1/2} c^* v^{1/2}``
form (Bard & Faulkner, *Electrochemical Methods*, 2nd ed., linear-sweep /
cyclic voltammetry chapter).  For an irreversible wave the coefficient
``0.4463`` is replaced by ``0.4958`` and ``n`` inside the square root by
``alpha n_a``; set ``reversible=False`` for that case.

Parameters
----------
n : int
    Electrons transferred.
A : float
    Electrode area (cm^2).
D : float
    Diffusion coefficient (cm^2/s).
c_bulk : float
    Bulk concentration (mol/cm^3).
v : float
    Sweep rate (V/s).
temperature : float
    Temperature (K).
reversible : bool
    If False, use the totally irreversible coefficient.
alpha : float
    Transfer coefficient (only used when ``reversible=False``).

Returns
-------
float
    Peak current magnitude (A).
```

### `sand_transition_time(n, A, D, c_bulk, i_applied)`

```
Sand equation -- transition time in chronopotentiometry.

.. math::
    \tau = \frac{\pi D (n F A c^*)^2}{4 i^2}

i.e. ``i \sqrt{\tau} = \tfrac12 n F A c^* \sqrt{\pi D}`` (Bard &
Faulkner, *Electrochemical Methods*, 2nd ed., chronopotentiometry chapter).
Validates Chapter 9.

Parameters
----------
n : int
    Electrons transferred.
A : float
    Electrode area (cm^2).
D : float
    Diffusion coefficient (cm^2/s).
c_bulk : float
    Bulk concentration (mol/cm^3).
i_applied : float
    Magnitude of the applied constant current (A).

Returns
-------
float
    Transition time tau (s).
```

### `surface_wave_peak_current(n, A, gamma, v, temperature=298.15)`

```
Peak current of a reversible surface-confined (adsorbed) redox couple.

For a strongly adsorbed, reversibly reacting monolayer the CV peak current
is

.. math::
    i_p = \frac{n^2 F^2 A \Gamma^* v}{4 R T}

(Bard & Faulkner, *Electrochemical Methods*, 2nd ed., chapter on
adsorbed / surface-confined species).  The peak is symmetric and grows
*linearly* with sweep rate ``v`` (contrast the ``sqrt(v)`` of a diffusing
species).  Validates Chapter 11.

Parameters
----------
n : int
    Electrons transferred.
A : float
    Electrode area (cm^2).
gamma : float
    Surface coverage ``Gamma*`` (mol/cm^2).
v : float
    Sweep rate (V/s).
temperature : float
    Temperature (K).

Returns
-------
float
    Surface-wave peak current (A).
```


## `serm.kinetics`

Shared electrode-kinetics and dimensionless-sweep primitives.

### `bv_surface_conc(c1, c2, xi, ks_star: 'float', alpha: 'float', tmp=None)`

```
Eliminated Butler--Volmer surface concentration ``c0``.

``c0 = (ks_star xi**(1-alpha) + 4 c1 - c2) tmp`` where ``c1, c2`` are the
first two interior nodes and ``tmp`` is :func:`bv_surface_factor` (recomputed
if not supplied).  This is the surface value eliminated from the one-sided
three-point flux balance (SERM Ch. 6).

Parameters
----------
c1, c2 : float or array_like
    Concentration at the first and second interior nodes.
xi : float or array_like
    Surface ratio ``exp[(nF/RT)(E - E0)]``.
ks_star : float
    Dimensionless standard rate constant.
alpha : float
    Transfer coefficient.
tmp : float or array_like, optional
    Precomputed :func:`bv_surface_factor`; computed from ``xi`` if omitted.
```

### `bv_surface_factor(xi, ks_star: 'float', alpha: 'float')`

```
Butler--Volmer surface-elimination factor ``tmp``.

``tmp = xi**alpha / (3 xi**alpha + ks_star (1 + xi))`` (SERM Ch. 6).  As
``ks_star -> inf`` this gives the Nernstian Dirichlet limit.

Parameters
----------
xi : float or array_like
    Surface ratio ``exp[(nF/RT)(E - E0)]``.
ks_star : float
    Dimensionless standard rate constant.
alpha : float
    Transfer coefficient.
```

### `f_thermal(temperature: 'float' = 298.15) -> 'float'`

```
Return ``f = F /(R T)`` (1/V), the inverse thermal voltage per electron.
```

### `ks_star_sweep(ks_dim: 'float', T: 'float', D_M: 'float', n: 'int') -> 'float'`

```
Dimensionless standard rate constant ``2 ks_dim sqrt(T /(D_M (n-1)))``.

The grid-scaled rate constant ``ksStar`` of SERM Chapters 6 and 15, where
``ks_dim`` is the dimensional standard rate constant ``k^o`` (cm/s), ``T`` the
total dimensionless sweep length and ``n`` the number of time steps.
```

### `space_points_6sigma(D_M: 'float', n: 'int', x_extent: 'float' = 6.0) -> 'int'`

```
Number of spatial nodes ``m = 1 + ceil(x_extent sqrt(D_M (n-1)))``.

The semi-infinite diffusion problems of SERM resolve the diffusion layer out
to ``x_extent`` diffusion lengths; with the dimensionless space step
``dx = 1/sqrt(D_M (n-1))`` that needs ``x_extent sqrt(D_M (n-1))`` steps.

Parameters
----------
D_M : float
    Model (dimensionless) diffusion coefficient.
n : int
    Number of time / potential grid points.
x_extent : float
    Domain length in diffusion lengths (6 in the source notebooks).

Returns
-------
int
```

### `triangular_sweep_potential(n: 'int', tau: 'float', T: 'float', upper_limit: 'float') -> 'np.ndarray'`

```
Dimensionless potential ``nF(E - E0)/RT`` along a triangular CV sweep.

Ramps down from ``upper_limit`` until the vertex at ``k = (n+1)/2`` and back
up, matching the ``cv2`` mapping of SERM (forward branch
``upper - (k-1) tau``; reverse branch ``upper - T + (k-1) tau``, with ``k``
1-based over ``1 .. n``).

Parameters
----------
n : int
    Number of potential steps.
tau : float
    Dimensionless step size in units of ``RT/nF``.
T : float
    Total dimensionless sweep length ``2(upper + |lower|)``.
upper_limit : float
    Dimensionless potential at the sweep ends.

Returns
-------
numpy.ndarray, shape (n,)
```


## B.2 Tiny runnable examples (one per module)

Each cell below exercises the module's core entry point with a small, fast input
so the example doubles as a smoke test.

### `serm` (top level) — explicit FD diffusion solve

In [3]:
from serm.grids import make_grid

# Small explicit FD grid: m space points, n time slices, D_M <= 0.5 for stability.
m, n, D_M = 25, 40, 0.45
c = make_grid(m, n)            # IC = 1 in bulk, surface BC = 0
c = serm.explicit_solve(c, D_M)
i = serm.electrode_current(c, D_M)
print("grid shape:", c.shape)
print("dimensionless current i[1:4]:", np.round(i[1:4], 4))
print("analytic Cottrell i[1:4]   :", np.round(serm.cottrell_dimensionless(n)[1:4], 4))

grid shape: (25, 40)
dimensionless current i[1:4]: [6.2839 2.5136 2.5607]
analytic Cottrell i[1:4]   : [3.5234 2.4914 2.0342]


### `serm.tridiagonal` — Thomas-algorithm solve

In [4]:
from serm.tridiagonal import tridiag_solve

# Solve a 4x4 tridiagonal system A x = b and check against a dense rebuild.
# Convention: sub/super diagonals have length n-1, main diagonal length n.
sub  = np.array([1.0, 1.0, 1.0])        # sub-diagonal  (x), length n-1
diag = np.array([4.0, 4.0, 4.0, 4.0])   # main diagonal (y), length n
sup  = np.array([1.0, 1.0, 1.0])        # super-diagonal(z), length n-1
b    = np.array([5.0, 6.0, 6.0, 5.0])
x = tridiag_solve(sub, diag, sup, b)

A = np.diag(diag) + np.diag(sup, 1) + np.diag(sub, -1)
print("tridiag_solve x:", np.round(x, 4))
print("max residual   :", np.max(np.abs(A @ x - b)))

tridiag_solve x: [1. 1. 1. 1.]
max residual   : 0.0


### `serm.filters` — smoothing a noisy signal

In [5]:
from serm.filters import moving_average, gaussian_kernel, convolution_filter

rng = np.random.default_rng(0)
sig = np.sin(np.linspace(0, 4*np.pi, 200)) + 0.3*rng.standard_normal(200)
ma  = moving_average(sig, 9)
gk  = gaussian_kernel(9)
cf  = convolution_filter(sig, 9)
print("moving_average out len:", np.asarray(ma).shape)
print("gaussian_kernel sum   :", round(float(np.sum(gk)), 6))
print("convolution_filter len:", np.asarray(cf).shape)

moving_average out len: (192,)
gaussian_kernel sum   : 1.0
convolution_filter len: (200,)


### `serm.grids` — dimensionless space grid

In [6]:
from serm.grids import space_points, dx_dimensionless, make_grid

D_M, n = 0.45, 50
m  = space_points(D_M, n)        # number of spatial points (an int)
dx = dx_dimensionless(D_M, n)    # dimensionless space step (a float)
g  = make_grid(m, n)             # IC = 1 everywhere; electrode row -> 0 for t>0
print("space_points (m)  :", m)
print("dx_dimensionless  :", round(float(dx), 5))
print("make_grid shape   :", g.shape)
print("electrode g[0,1]  :", g[0, 1], "| bulk g[-1,1]:", g[-1, 1], "| IC g[1,0]:", g[1, 0])

space_points (m)  : 30
dx_dimensionless  : 0.21296
make_grid shape   : (30, 50)
electrode g[0,1]  : 0.0 | bulk g[-1,1]: 1.0 | IC g[1,0]: 1.0


### `serm.plotting` — current plot helper (headless)

In [7]:
from serm.plotting import plot_current

n = 50
tau = np.linspace(0, 1, n)
i_sim = serm.electrode_current(serm.explicit_solve(make_grid(25, n), 0.45), 0.45)
fig, ax = plt.subplots()
plot_current(tau, i_sim, i_cottrell=serm.cottrell_dimensionless(n), ax=ax)
print("axes lines drawn:", len(ax.get_lines()))
plt.close(fig)

axes lines drawn: 2


### `serm.waveforms` — linear and cyclic potential programs

In [8]:
from serm.waveforms import linear_sweep, cyclic_sweep, nernst_theta

t1, E1 = linear_sweep(E_start=0.2, E_end=-0.2, v=0.1, n_points=11)
t2, E2 = cyclic_sweep(E_start=0.2, E_switch=-0.2, v=0.1, n_points=11)
theta  = nernst_theta(E=np.array([0.05, 0.0, -0.05]), E0=0.0)
print("linear_sweep  E[:3] :", np.round(np.asarray(E1)[:3], 4))
print("cyclic_sweep  E min/max:", round(float(np.min(E2)),4), round(float(np.max(E2)),4))
print("nernst_theta        :", np.round(np.asarray(theta), 4))

linear_sweep  E[:3] : [0.2  0.16 0.12]
cyclic_sweep  E min/max: -0.2 0.2
nernst_theta        : [7.0012 1.     0.1428]


### `serm.echem` — analytic diagnostics (Cottrell, Randles-Sevcik, Levich)

In [9]:
from serm.echem import cottrell_current, randles_sevcik_peak_current, levich_current

# SI-ish example values.
n_e, A, D, c_bulk = 1, 1.0e-4, 1.0e-9, 1.0   # mol/m^3 etc. (illustrative)
t = np.array([0.1, 1.0, 10.0])
print("echem.cottrell_current(t):", cottrell_current(t, n_e, A, D, c_bulk))
print("randles_sevcik (v=0.1)   :", randles_sevcik_peak_current(n_e, A, D, c_bulk, v=0.1))
print("levich (omega=100,nu=1e-6):", levich_current(n_e, A, D, c_bulk, omega=100.0, nu=1.0e-6))

echem.cottrell_current(t): [5.44360193e-04 1.72141808e-04 5.44360193e-05]
randles_sevcik (v=0.1)   : 0.0002686484453190527
levich (omega=100,nu=1e-6): 0.0005982090591440004


<!-- nav-footer -->

---

| | |
|:--|--:|
| [← Appendix A — A Python Refresher for Electrochemical Simulation](A_appendix_a_python_refresher.ipynb) | &nbsp; |

[Contents (README)](../README.md)